# SSD300 (VGG-16) — evaluation

Scores the checkpoint written by [`ssd300_train.ipynb`](ssd300_train.ipynb) (`models/ssd300/weights/best.pt`) on the test split.

Test images are only resized to 640×640 (no augmentation). Predictions, ground truth and per-image latency are collected in the
`EvaluationFramework` (see [`../example.ipynb`](../example.ipynb) for the metric definitions) and saved as JSON for the cross-model comparison.

In [1]:
import time

import torch
from torchvision.models.detection import ssd300_vgg16

from ppe_detection_evaluation import SSD300, EvaluationFramework
from ppe_detection_evaluation.classes import DEFAULT_CLASS_NAMES
from ppe_detection_evaluation.dataset.core import check_dataset, list_images, load_eval_example, xyxy_to_cxcywh
from ppe_detection_evaluation.experiment import DATA_ROOT, IMAGE_SIZE, MODELS_DIR, RESULTS_DIR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
NUM_CLASSES = len(DEFAULT_CLASS_NAMES) + 1  # + background at index 0
check_dataset(DATA_ROOT)  # the training notebook downloads the dataset

{'train': 1132, 'valid': 143, 'test': 141}

In [2]:
# Same architecture as the training notebook. No pretrained weights are downloaded: the trained weights all come
# from the checkpoint.
model = ssd300_vgg16(
    weights=None,
    weights_backbone=None,
    num_classes=NUM_CLASSES,
    score_thresh=0.001,
)

checkpoint = MODELS_DIR / "ssd300" / "weights" / "best.pt"
model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
model = model.to(DEVICE).eval()

In [3]:
def sync():
    # Wait for queued GPU work, so the timer measures the forward pass rather than just its launch.
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


fw = EvaluationFramework(SSD300)
with torch.inference_mode():
    for path in list_images(DATA_ROOT, "test"):
        image, gt_boxes, gt_labels = load_eval_example(path)
        tensor = torch.from_numpy(image).permute(2, 0, 1).float().div(255).to(DEVICE)

        sync()
        start = time.perf_counter()
        out = model([tensor])[0]
        sync()
        latency_ms = (time.perf_counter() - start) * 1000

        fw.add(
            path.stem,
            out["boxes"].cpu().numpy(),
            out["scores"].cpu().numpy(),
            out["labels"].cpu().numpy(),
            xyxy_to_cxcywh(gt_boxes, IMAGE_SIZE, IMAGE_SIZE),
            gt_labels,
            latency_ms=latency_ms,
        )

results = fw.evaluate()
fw.save(RESULTS_DIR / "ssd300_test.json")

PosixPath('/home/llravelo/Documents/projects/42177-image-processing-ppe-detection/results/ssd300_test.json')

In [4]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])

mAP@[.5:.95] 0.3060   mAP@.5 0.6104
PPE mAP@.5 0.5633   small PPE mAP@.5 0.0718
violation mean: {'precision_at_recall': 0.0, 'recall_at_fppi': 0.2826923076923077, 'lamr': 0.6969900192071127, 'max_recall': 0.5923076923076923, 'max_f1': 0.40349315830459453}
latency: {'num_images': 141, 'mean_ms': 26.700887524778068, 'std_ms': 35.06051879525601, 'p50_ms': 9.003241000755224, 'p95_ms': 60.12782799916749, 'fps': 37.451938594625865}
